<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part A: Foundations and Data Exploration</h2>
<h2>Notebook A06: Evaluating Models</h2>
</div>

Notebook A05 compared five baselines using a single number on a single split. That was enough to rank
them, but it hid almost everything that can go wrong in an evaluation.

This notebook takes evaluation seriously. It asks whether the series is predictable in the first place,
works through the error metrics and what each one quietly rewards, shows why one train/test split gives
you a number with more uncertainty than you would guess, and finishes with the diagnostic that tells you
whether there is anything left to extract.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [Is This Series Forecastable at All](#2.-Is-This-Series-Forecastable-at-All)
3. [The Standard Error Metrics](#3.-The-Standard-Error-Metrics)
4. [Percentage and Scaled Errors](#4.-Percentage-and-Scaled-Errors)
5. [One Split Is Not Enough](#5.-One-Split-Is-Not-Enough)
6. [Residual Diagnostics](#6.-Residual-Diagnostics)
7. [An Evaluation Checklist](#7.-An-Evaluation-Checklist)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL

import nb_config

sns.set_theme(style="whitegrid")

We continue with the CDC temperature series and the baselines from Notebook
[A05](./A05_Forecasting_baselines.ipynb), so that the two notebooks can be read as one story. The
baseline functions are repeated here to keep this notebook self-contained.

In [ ]:
temperatures = pd.read_parquet(nb_config.CDC_TEMP_PATH)
series = temperatures["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12

train = series.iloc[:-TEST_MONTHS]
test = series.iloc[-TEST_MONTHS:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()}  ({len(train)} months)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()}  ({len(test)} months)")

In [ ]:
def forecast_index(train, horizon):
    """The dates a forecast covers, continuing the training index."""
    return pd.date_range(
        start=train.index[-1] + train.index.freq,
        periods=horizon,
        freq=train.index.freq,
    )


def naive_forecast(train, horizon):
    return pd.Series(train.iloc[-1], index=forecast_index(train, horizon), name="Naive")


def seasonal_naive_forecast(train, horizon, season_length=SEASON_LENGTH):
    last_cycle = train.iloc[-season_length:].to_numpy()
    values = [last_cycle[i % season_length] for i in range(horizon)]
    return pd.Series(values, index=forecast_index(train, horizon), name="Seasonal naive")


def mean_forecast(train, horizon):
    return pd.Series(train.mean(), index=forecast_index(train, horizon), name="Mean")


forecasts = {
    "Seasonal naive": seasonal_naive_forecast(train, TEST_MONTHS),
    "Mean": mean_forecast(train, TEST_MONTHS),
    "Naive": naive_forecast(train, TEST_MONTHS),
}

pd.DataFrame({"Actual": test, **forecasts}).head().round(2)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Is-This-Series-Forecastable-at-All">2. Is This Series Forecastable at All</h3>
</div>

Before comparing models, it is worth asking whether the series can be predicted at all. A series that is
essentially noise will defeat every method you throw at it, and knowing that early saves weeks of work.
None of the measures below gives a yes-or-no answer, but together they tell you what kind of series you
are dealing with.

#### Coefficient of variation

The ratio of the standard deviation to the mean. A low value means the series stays close to its own
average and is, other things equal, easier to forecast.

It comes with a trap that is easy to fall into.

In [ ]:
celsius = series
kelvin = series + 273.15  # the same physical measurements, different zero point

print(f"CV in Celsius: {celsius.std() / celsius.mean():.3f}")
print(f"CV in Kelvin:  {kelvin.std() / kelvin.mean():.3f}")

The same series, the same physical variability, and a coefficient of variation that differs by a factor of
more than thirty.

The reason is that the CV divides by the mean, which only means something when zero means "none of it".
That holds for sales, energy consumption, or website visits. It does not hold for temperature in Celsius,
where zero is an arbitrary point on the scale, nor for anything that can go negative.

**Use the CV on ratio-scaled data only.** For this series it is the wrong tool, and we report it here
mainly so that you recognise the trap when you meet it.

#### Residual variability

A more robust question: once the patterns you can model are removed, how much is left? We decompose the
series into trend, season, and remainder, and compare the size of the remainder against the size of the
original.

In [ ]:
decomposition = STL(train, period=SEASON_LENGTH, robust=True).fit()

residual_share = decomposition.resid.std() / train.std()

print(f"Standard deviation of the series:    {train.std():.2f} °C")
print(f"Standard deviation of the remainder: {decomposition.resid.std():.2f} °C")
print(f"Residual variability:                {residual_share:.3f}")
print(f"Explained by trend and season:       {1 - residual_share:.1%}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)

for ax, (values, label) in zip(
    axes,
    [
        (train, "Observed"),
        (decomposition.trend, "Trend"),
        (decomposition.seasonal, "Season"),
        (decomposition.resid, "Remainder"),
    ],
):
    ax.plot(values["1990":], linewidth=0.9, color="steelblue")
    ax.set_ylabel(label)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].set_title("STL decomposition from 1990", fontsize=14, fontweight="bold")
axes[-1].set_xlabel("Date")

plt.tight_layout()
plt.show()

Roughly three quarters of the variation is trend and season, both of which a model can learn. The
remaining quarter is what any forecast will be competing against, and it puts a floor under the error
that no model can go below.

Unlike the CV, this measure does not care about the units or where the zero sits.

#### Sample entropy

Entropy measures how repetitive a series is. **Sample entropy** asks: if two stretches of the series look
alike for *m* steps, how often do they still look alike at step *m + 1*? A structured series says "very
often" and scores low. Noise says "no more often than chance" and scores high.

We compare the series against pure noise, and against its own STL remainder, to calibrate what the
numbers mean.

In [ ]:
def sample_entropy(values, m=2, tolerance=0.2):
    """Sample entropy of a series, computed on standardised values.

    `m` is the pattern length and `tolerance` the distance below which two
    patterns count as a match, in standard deviations.
    """
    values = np.asarray(values, dtype=float)
    values = (values - values.mean()) / values.std()
    n = len(values)

    def count_matches(length):
        patterns = np.array([values[i:i + length] for i in range(n - length + 1)])
        total = 0
        for i, pattern in enumerate(patterns):
            distances = np.max(np.abs(patterns - pattern), axis=1)
            total += np.sum(distances <= tolerance) - 1  # exclude the self-match
        return total

    matches_long = count_matches(m + 1)
    matches_short = count_matches(m)

    if matches_long == 0 or matches_short == 0:
        return np.inf
    return float(-np.log(matches_long / matches_short))

In [ ]:
# Sample entropy is O(n^2), so we use the most recent 50 years
recent = train.iloc[-600:]
white_noise = pd.Series(np.random.default_rng(0).normal(size=len(recent)))

print(f"Temperature series: {sample_entropy(recent.values):.3f}")
print(f"STL remainder:      {sample_entropy(decomposition.resid.iloc[-600:].values):.3f}")
print(f"White noise:        {sample_entropy(white_noise.values):.3f}")

The ordering is exactly what it should be. The raw series is the most predictable, because its seasonal
cycle repeats. Its remainder scores higher, since the repeating part has been taken out. Pure noise scores
highest of all.

Entropy has no absolute scale, so a single value tells you nothing. It is a comparative tool: between
series, or between a series and its own residuals.

#### The Kaboudan metric

A more direct question: is this series more predictable than a scrambled version of itself? Forecast the
real series, then shuffle it in blocks to destroy the temporal structure, forecast that, and compare the
sum of squared errors:

$$K = 1 - \frac{SSE_{\text{original}}}{SSE_{\text{shuffled}}}$$

A value near 1 means the temporal structure is carrying real information. A value near 0 means the order
of the observations does not help, which is the signature of noise.

In [ ]:
def seasonal_naive_sse(values, season_length=SEASON_LENGTH):
    """Sum of squared one-cycle-ahead errors."""
    values = np.asarray(values, dtype=float)
    return float(np.sum((values[season_length:] - values[:-season_length]) ** 2))


def kaboudan_metric(values, block_size=5, n_shuffles=20, seed=0):
    """How much better a forecast does on the real series than on a shuffled one."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)

    n_blocks = len(values) // block_size
    blocks = values[:n_blocks * block_size].reshape(n_blocks, block_size)

    shuffled_sse = [
        seasonal_naive_sse(blocks[rng.permutation(n_blocks)].reshape(-1))
        for _ in range(n_shuffles)
    ]

    return 1 - seasonal_naive_sse(values) / np.mean(shuffled_sse)

In [ ]:
print(f"Temperature series: {kaboudan_metric(recent.values):.3f}")
print(f"White noise:        {kaboudan_metric(white_noise.values):.3f}")

0.93 against roughly zero. Scrambling the temperature series destroys almost all of its predictability;
scrambling noise changes nothing, because there was nothing there to destroy.

A slightly negative value, as here, is not a meaningful result: it just means the shuffled version
happened to be forecast a shade better than the original. Anything at or below zero reads the same way,
as no predictable structure.

> **Watch the block size.** We shuffle in blocks of 5 months. Had we used blocks of 12, each block would
> be a whole year and the shuffle would leave every seasonal cycle perfectly intact, so the metric would
> report almost no predictability where there is plenty. Never let the block length be a multiple of the
> season.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-The-Standard-Error-Metrics">3. The Standard Error Metrics</h3>
</div>

Now to measuring forecasts. Write $y_i$ for the actual value and $\hat{y}_i$ for the forecast. The four
metrics you will meet everywhere:

| Metric | Definition | In words |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum |y_i - \hat{y}_i|$ | Average error size, in the units of the data |
| **MSE** | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ | Average squared error, in squared units |
| **RMSE** | $\sqrt{\text{MSE}}$ | Back in the original units, but still squares first |
| **Bias** | $\frac{1}{n}\sum (\hat{y}_i - y_i)$ | Direction of the error: positive means over-forecasting |

The difference that matters in practice is between MAE and RMSE. Squaring makes RMSE much more sensitive
to a few large errors, so a model with consistent small errors beats a model that is usually perfect but
occasionally far off. Which behaviour you prefer is a question about your problem, not about statistics:
for inventory, one catastrophic miss may cost more than a hundred small ones, and RMSE captures that.

Bias is the one people forget, and it is the only one of the four that can be zero while the forecast is
useless. It answers a different question: not how big the errors are, but whether they systematically
lean one way.

In [ ]:
def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(actual - forecast)))


def mean_squared_error(actual, forecast):
    return float(np.mean((actual - forecast) ** 2))


def root_mean_squared_error(actual, forecast):
    return float(np.sqrt(mean_squared_error(actual, forecast)))


def forecast_bias(actual, forecast):
    return float(np.mean(forecast - actual))


metrics = pd.DataFrame(
    [
        {
            "Forecast": name,
            "MAE": mean_absolute_error(test.values, forecast.values),
            "MSE": mean_squared_error(test.values, forecast.values),
            "RMSE": root_mean_squared_error(test.values, forecast.values),
            "Bias": forecast_bias(test.values, forecast.values),
        }
        for name, forecast in forecasts.items()
    ]
).sort_values("MAE").reset_index(drop=True)

metrics.round(2)

The ranking is the same under MAE, MSE and RMSE here, which is common when one method is clearly better
than the others. Note that RMSE is larger than MAE for every forecast: that gap is a sign that the errors
are uneven, since the two are equal only when every error has the same size.

The bias column adds something the others cannot. The naive forecast has a bias of +7.97 °C: it does not
merely miss, it misses *upwards* almost every month, because it repeats an August value all through the
following winter. A forecast like that is broken in a specific, fixable way, and no amount of looking at
MAE would tell you which direction to fix it in.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

for name, forecast in forecasts.items():
    ax.plot(forecast.index, forecast.values - test.values, marker="o", markersize=4,
            linewidth=1.2, label=name)

ax.axhline(0, color="black", linewidth=1.0)
ax.set_title("Forecast errors over the test period", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Forecast - actual (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Plotting the errors over time makes the bias visible: the naive forecast sits almost entirely above the
zero line, and swings with the season it failed to capture. The seasonal naive errors hug the line and
change sign, which is what an unbiased forecast looks like.

Always plot the errors. A single summary number cannot show you a pattern in them, and a pattern in the
errors is the most actionable thing an evaluation can give you.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Percentage-and-Scaled-Errors">4. Percentage and Scaled Errors</h3>
</div>

MAE and RMSE are in the units of the data, which makes them easy to interpret but impossible to compare
across series. An MAE of 1.74 °C and an MAE of 1,200 euros cannot be ranked. Two families of metrics try
to fix this.

#### Percentage errors

**MAPE** expresses each error as a percentage of the actual value:

$$\text{MAPE} = \frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

It is the most widely used metric in business forecasting, and the most widely misused.

In [ ]:
def mean_absolute_percentage_error(actual, forecast):
    return float(np.mean(np.abs((actual - forecast) / actual)) * 100)


mape_values = {
    name: mean_absolute_percentage_error(test.values, forecast.values)
    for name, forecast in forecasts.items()
}

for name, value in mape_values.items():
    print(f"{name:<16} MAE={mean_absolute_error(test.values, forecasts[name].values):5.2f} °C   "
          f"MAPE={value:7.1f}%")

A seasonal naive forecast that is off by 1.74 °C on average scores a MAPE of 36%. That sounds like a poor
forecast. It is not: the metric has broken.

In [ ]:
errors = pd.DataFrame({
    "Actual": test,
    "Forecast": forecasts["Seasonal naive"],
})
errors["Absolute error"] = (errors["Actual"] - errors["Forecast"]).abs()
errors["Percentage error"] = errors["Absolute error"] / errors["Actual"].abs() * 100

print("The months that dominate the MAPE:")
print(errors.nlargest(4, "Percentage error").round(2).to_string())

Two winter months sit near 1 °C. An error of two or three degrees there, unremarkable in absolute terms
and no worse than the forecast makes in July, becomes a percentage error above 200%. Those two months
alone drag the average far above anything the forecast deserves.

**MAPE is unusable when the actual values can be near zero**, and undefined when they are exactly zero. It
also punishes over-forecasting more than under-forecasting, because the denominator is the actual value.
Reserve it for strictly positive series that stay well away from zero, such as demand or traffic.

#### Scaled errors

**MASE** takes a different route. Instead of dividing by the actual values, it divides by the error of a
naive forecast computed *in-sample*:

$$\text{MASE} = \frac{\text{MAE of the forecast}}{\text{MAE of a seasonal naive forecast on the training data}}$$

This gives a number with an unambiguous reading: **below 1 is better than the naive benchmark, above 1 is
worse**. It is defined whenever the series is not constant, it works with zeros and negatives, and it is
comparable across series of any scale.

In [ ]:
def mean_absolute_scaled_error(actual, forecast, train, season_length=SEASON_LENGTH):
    """MAE divided by the in-sample seasonal naive MAE."""
    train_values = np.asarray(train, dtype=float)
    scale = np.mean(np.abs(train_values[season_length:] - train_values[:-season_length]))
    return mean_absolute_error(actual, forecast) / scale


scale = np.mean(np.abs(train.values[SEASON_LENGTH:] - train.values[:-SEASON_LENGTH]))
print(f"In-sample seasonal naive MAE (the scale): {scale:.2f} °C\n")

metrics["MAPE"] = [mape_values[name] for name in metrics["Forecast"]]
metrics["MASE"] = [
    mean_absolute_scaled_error(test.values, forecasts[name].values, train.values)
    for name in metrics["Forecast"]
]

metrics[["Forecast", "MAE", "RMSE", "Bias", "MAPE", "MASE"]].round(2)

The seasonal naive forecast scores a MASE of 0.85. Slightly below 1, which says it does a little better
over this particular two-year window than the same method does on average across the training data. The
mean forecast scores 2.96 and the naive forecast 3.99: three to four times worse than the benchmark.

Read against MAPE, the contrast is instructive. MAPE called the seasonal naive forecast 36% wrong and the
mean forecast 113% wrong, numbers that are hard to act on and were driven by two cold months. MASE puts
them at 0.85 and 2.96, which say exactly what you want to know.

**Exercise.** Compute all five metrics for a forecast that always predicts the training median. Where does it rank? Then construct a deliberately biased forecast by adding 3 °C to the seasonal naive values, and check which metrics notice and which do not.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-One-Split-Is-Not-Enough">5. One Split Is Not Enough</h3>
</div>

Every number so far came from one split at one point in time. That makes them estimates, and estimates
have uncertainty. If the final two years happened to be unusually mild, every model looks better than it
is, and nothing in the numbers would reveal it.

**Rolling-origin evaluation**, also called backtesting or walk-forward validation, repeats the exercise
at several origins. Each time you train on everything up to a cut-off, forecast a fixed horizon, score
it, then move the cut-off forward and do it again. Instead of one number you get a distribution.

In [ ]:
def rolling_origin_evaluation(series, forecast_functions, horizon, n_origins, step):
    """Score each forecast function at several cut-off points.

    The origins end `horizon` steps before the end of the series and walk
    backwards in jumps of `step`.
    """
    results = []

    for k in range(n_origins):
        end = len(series) - horizon - (n_origins - 1 - k) * step
        train_slice = series.iloc[:end]
        test_slice = series.iloc[end:end + horizon]

        for name, make_forecast in forecast_functions.items():
            forecast = make_forecast(train_slice, horizon)
            results.append({
                "Origin": train_slice.index[-1].date(),
                "Forecast": name,
                "MAE": mean_absolute_error(test_slice.values, forecast.values),
            })

    return pd.DataFrame(results)


rolling = rolling_origin_evaluation(
    series,
    {
        "Seasonal naive": seasonal_naive_forecast,
        "Mean": mean_forecast,
        "Naive": naive_forecast,
    },
    horizon=12,
    n_origins=10,
    step=6,
)

by_origin = rolling.pivot(index="Origin", columns="Forecast", values="MAE")
by_origin.round(2)

In [ ]:
summary = by_origin.agg(["mean", "std", "min", "max"]).T
summary["Single split"] = [
    mean_absolute_error(test.values, forecasts[name].values) for name in summary.index
]

summary.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

sns.boxplot(data=rolling, x="Forecast", y="MAE", ax=ax, width=0.5)
sns.stripplot(data=rolling, x="Forecast", y="MAE", ax=ax, color="black", size=5, alpha=0.7)

ax.set_title("MAE across 10 forecast origins", fontsize=14, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("MAE (°C)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The seasonal naive forecast averages 1.63 °C across the ten origins, but the individual results range
from 1.03 to 2.08. That is a spread of a full degree on a method with no parameters and nothing to tune.
None of it is the method changing; it is the difference between an easy year and a hard one.

This is the practical lesson. A difference of 0.2 °C between two models on a single split is well inside
this noise, and choosing between them on that basis is guesswork. Run several origins before you believe a
ranking, and quote the spread alongside the mean.

Note also how stable the mean forecast is by comparison. A forecast that ignores recent data entirely is
insensitive to which window you test it on, which is consistency without merit.

**Exercise.** Re-run the rolling evaluation with `horizon=1` instead of 12. Do the errors get smaller, and does the gap between the three methods change? What does that tell you about comparing forecasts made at different horizons?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Residual-Diagnostics">6. Residual Diagnostics</h3>
</div>

A good forecast leaves behind residuals that look like noise. If a pattern remains in the errors, the
model has missed something that is still there to be used, and a better model exists.

Two checks. The **ACF of the residuals** shows whether errors at nearby times are related, and the
**Ljung-Box test** puts a p-value on it. The null hypothesis is that the residuals are independent, so a
small p-value means there is structure left.

In [ ]:
# One-step-ahead residuals of the seasonal naive method, over the whole series
residuals = (series - series.shift(SEASON_LENGTH)).dropna()

print(f"Residual mean: {residuals.mean():.3f} °C")
print(f"Residual std:  {residuals.std():.3f} °C")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(residuals["1990":], color="steelblue", linewidth=0.8)
axes[0].axhline(0, color="black", linewidth=1.0)
axes[0].set_title("Seasonal naive residuals (from 1990)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual (°C)")

plot_acf(residuals, lags=36, ax=axes[1], color="steelblue",
         vlines_kwargs={"colors": "steelblue"})
axes[1].set_title("ACF of the residuals", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Lag (months)")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
ljung_box = acorr_ljungbox(residuals, lags=[12, 24, 36], return_df=True)
ljung_box.round(4)

The residual mean is essentially zero, so the method is unbiased. But the ACF has clear spikes well
outside the confidence band, and the Ljung-Box p-values are indistinguishable from zero at every lag
tested. The residuals are emphatically not noise.

That is a useful finding rather than a disappointment. It says the seasonal naive forecast is leaving
usable structure on the table, which is precisely why the exponential smoothing models in Notebook A05
managed to beat it, and why the models in Part B will do better still.

When residual diagnostics come back clean, you have reached the noise floor of the series and further
modelling effort will not pay.

**Exercise.** Fit the Holt-Winters model from Notebook A05 (`ExponentialSmoothing` with additive trend and season) and run the same two diagnostics on `fitted.resid`. Is its ACF cleaner than the seasonal naive one? Does Ljung-Box still reject independence?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-An-Evaluation-Checklist">7. An Evaluation Checklist</h3>
</div>

Pulling the notebook together, in the order you would actually work through it:

**1. Check the series is worth forecasting.** Residual variability and entropy, compared against noise.
If almost nothing is left after removing trend and season, no model will save you.

**2. Split on time, never at random.** Hold out at least one full seasonal cycle, and at least as long as
the horizon you care about.

**3. Always compute a baseline.** A metric without a reference point carries no information.

**4. Pick metrics that suit the data.** MAE for a readable average, RMSE when large errors are
disproportionately costly, MASE to compare across series, bias always. Avoid MAPE unless the data is
strictly positive and far from zero.

**5. Plot the errors.** Summary numbers hide patterns; patterns tell you what to fix.

**6. Use several origins.** One split gives one draw from a distribution that is wider than you expect.

**7. Check the residuals.** Structure left in the errors means a better model is available.

A model is not "good" because its MAE is low. It is good when it beats a sensible baseline, by a margin
that survives being tested at several origins, with residuals that no longer contain a pattern, on a
metric that suits the decision the forecast supports.

---

That completes Part A. You can load and inspect a series, deal with gaps and outliers, build baselines
that are hard to beat, and evaluate a forecast in a way that will not mislead you.

Part B turns to the statistical models that have dominated forecasting for fifty years, starting with the
exponential smoothing family we have already borrowed from twice:
[B01 - Exponential Smoothing Models](./B01_Exponential_smoothing_models.ipynb).

**Solutions.** Worked answers to the 3 exercises above, with the reasoning behind them, are in
[A06_Evaluating_models_solutions.ipynb](../solutions/A06_Evaluating_models_solutions.ipynb). Try each one yourself first.
